In [7]:
import pandas as pd
import numpy as np
from entsoe import EntsoePandasClient

In [8]:
API_KEY = "nicetrystevenfry"
COUNTRY_CODE = "DE_LU"

client = EntsoePandasClient(api_key=API_KEY)

In [13]:
client = EntsoePandasClient(api_key=API_KEY)

start = pd.Timestamp("2024-01-01 00:00", tz="Europe/Berlin")
end = pd.Timestamp("2026-06-30 23:00", tz="Europe/Berlin")

print(f"Lade Daten von {start} bis {end} für Gebotszone {COUNTRY_CODE}...\n")


# A) Day-Ahead Price (EUR/MWh)
try:
    df_prices = client.query_day_ahead_prices(COUNTRY_CODE, start=start, end=end)
    df_prices = df_prices.to_frame(name="energy_prices")
except Exception as e:
    print(f"Fehler beim Laden der Preise: {e}")
    df_prices = pd.DataFrame()

# B) Total Load (MW)
try:
    df_load = client.query_load(COUNTRY_CODE, start=start, end=end)
    if isinstance(df_load, pd.Series):
        df_load = df_load.to_frame(name="total_load")
    else:
        df_load.columns = ["total_load"]
except Exception as e:
    print(f"Fehler beim Laden der Lastdaten: {e}")
    df_load = pd.DataFrame()

# C) Day-Ahead Solar & Wind Prediction (MW)
try:
    df_forecast = client.query_wind_and_solar_forecast(COUNTRY_CODE, start=start, end=end)
except Exception as e:
    print(f"Fehler beim Laden der Erzeugungsprognose: {e}")
    df_forecast = pd.DataFrame()

if "Wind Onshore" in df_forecast.columns or "Wind Offshore" in df_forecast.columns:
    wind_cols = [c for c in ["Wind Onshore", "Wind Offshore"] if c in df_forecast.columns]
    df_forecast["wind_forecast"] = df_forecast[wind_cols].sum(axis=1)

if "Solar" in df_forecast.columns:
    df_forecast.rename(columns={"Solar": "day_ahead_solar_forecast"}, inplace=True)

df_combined = pd.concat([df_prices, df_load, df_forecast], axis=1, sort=False)

target_cols = ["energy_prices", "total_load", "day_ahead_solar_forecast", "wind_forecast"]
existing_cols = [col for col in target_cols if col in df_combined.columns]
df_combined = df_combined[existing_cols]
print("Done.")

Lade Daten von 2024-01-01 00:00:00+01:00 bis 2026-06-30 23:00:00+02:00 für Gebotszone DE_LU...

Done.


In [12]:
df_hourly = df_combined.resample("1h").mean()
# Make hours and years cyclical for deep learning
hours = df_hourly.index.hour
df_hourly["hour_sin"] = np.sin(2 * np.pi * hours / 24)
df_hourly["hour_cos"] = np.cos(2 * np.pi * hours / 24)

days_in_year = np.where(df_hourly.index.is_leap_year, 366, 365)
day_of_year = df_hourly.index.dayofyear
df_hourly["day_of_year_sin"] = np.sin(2 * np.pi * day_of_year / days_in_year)
df_hourly["day_of_year_cos"] = np.cos(2 * np.pi * day_of_year / days_in_year)

#One Hot Encoding for Weekday and Montht
weekday_dummies = pd.get_dummies(
    df_hourly.index.dayofweek, prefix="weekday", dtype=int
)
weekday_dummies.index = df_hourly.index

month_dummies = pd.get_dummies(
    df_hourly.index.month, prefix="month", dtype=int
)
month_dummies.index = df_hourly.index

df_prepared = pd.concat([df_hourly, weekday_dummies, month_dummies], axis=1)
df_prepared = df_prepared.dropna()
print(df_prepared.head())
df_prepared.to_csv('entsoe_hourly_data_de_lu.csv')

                           energy_prices  total_load  \
2024-01-01 00:00:00+01:00           0.10  40584.9175   
2024-01-01 01:00:00+01:00           0.01  39221.1400   
2024-01-01 02:00:00+01:00           0.00  38238.5900   
2024-01-01 03:00:00+01:00          -0.01  37508.4400   
2024-01-01 04:00:00+01:00          -0.03  37134.7750   

                           day_ahead_solar_forecast  wind_forecast  hour_sin  \
2024-01-01 00:00:00+01:00                       0.0     35376.6625  0.000000   
2024-01-01 01:00:00+01:00                       0.0     34897.8200  0.258819   
2024-01-01 02:00:00+01:00                       0.0     34576.8850  0.500000   
2024-01-01 03:00:00+01:00                       0.0     34137.8950  0.707107   
2024-01-01 04:00:00+01:00                       0.0     33765.0500  0.866025   

                           hour_cos  day_of_year_sin  day_of_year_cos  \
2024-01-01 00:00:00+01:00  1.000000         0.017166         0.999853   
2024-01-01 01:00:00+01:00  0.965926 